# Notebook 03 — Quantum Risk Analysis & Option Pricing

**Papers implemented:**
- Woerner & Egger (2019) *Quantum risk analysis* — Nature npj Quantum Information
- Stamatopoulos et al. (2020) *Option pricing using quantum computers* — Quantum 4, 291
- Rebentrost et al. *Quantum computational finance: Monte Carlo pricing of financial derivatives*

**Key result:** Quantum Amplitude Estimation (QAE) achieves **O(1/M)** precision
versus classical Monte Carlo **O(1/√N)** — a quadratic speedup.

**Learning path:** This notebook is Step 3 of 5. It bridges classical simulation
(NB01–02) to quantum hardware advantage (NB04–05).

## 0. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

COLORS = {
    'classical': '#2196F3',
    'quantum':   '#E91E63',
    'bs':        '#4CAF50',
    'cvar':      '#FF9800',
    'neutral':   '#9E9E9E',
}

print("✓ Imports loaded")
print("Implementing: Woerner & Egger (2019), Stamatopoulos et al. (2020), Rebentrost et al.")

## 1. Black-Scholes Closed-Form Baseline

The Black-Scholes formula (1973) provides the theoretical benchmark for all
option pricing methods. We implement it here as the ground truth against which
quantum and classical Monte Carlo methods are measured.

$$C(S,K,T,r,\sigma) = S\,N(d_1) - K e^{-rT}\,N(d_2)$$

$$d_1 = \frac{\ln(S/K)+(r+\sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
def black_scholes(S, K, T, r, sigma, option='call'):
    """
    Black-Scholes closed-form option price.

    Parameters
    ----------
    S     : float  Current stock price
    K     : float  Strike price
    T     : float  Time to maturity (years)
    r     : float  Risk-free rate (annual)
    sigma : float  Volatility (annual)
    option: str    'call' or 'put'

    Returns
    -------
    float : Option price

    Reference: Black & Scholes (1973), Merton (1973)
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:  # put
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return price


def bs_greeks(S, K, T, r, sigma):
    """Black-Scholes Greeks for a call option."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
             - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    vega  = S * norm.pdf(d1) * np.sqrt(T) / 100
    return {'delta': delta, 'gamma': gamma, 'theta': theta, 'vega': vega}


# ── Standard parameters (used throughout notebook) ──────────────────────────
S0    = 100.0   # Spot price
K     = 105.0   # Strike (slightly OTM call)
T     = 1.0     # 1 year
r     = 0.05    # 5% risk-free
sigma = 0.20    # 20% vol

bs_call = black_scholes(S0, K, T, r, sigma, 'call')
bs_put  = black_scholes(S0, K, T, r, sigma, 'put')
greeks  = bs_greeks(S0, K, T, r, sigma)

print(f"Black-Scholes Call: ${bs_call:.4f}")
print(f"Black-Scholes Put:  ${bs_put:.4f}")
print(f"Put-Call Parity check: {abs(bs_call - bs_put - S0 + K*np.exp(-r*T)):.2e}")
print()
print("Greeks:")
for g, v in greeks.items():
    print(f"  {g:6s}: {v:.4f}")

### 1.1 Option Payoff Diagrams

Visualizing the payoff structure explains why quantum amplitude estimation
targets these functions — the non-linear kink at the strike is the source
of pricing difficulty.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

S_range = np.linspace(60, 150, 400)

# ── Panel 1: Payoff at expiry ──────────────────────────────────────────────
ax = axes[0]
call_payoff = np.maximum(S_range - K, 0)
put_payoff  = np.maximum(K - S_range, 0)
ax.plot(S_range, call_payoff, color=COLORS['quantum'], lw=2, label='Call payoff')
ax.plot(S_range, put_payoff,  color=COLORS['classical'], lw=2, label='Put payoff')
ax.axvline(K, color='gray', ls='--', alpha=0.6, label=f'Strike K={K}')
ax.axvline(S0, color='black', ls=':', alpha=0.5, label=f'Spot S={S0}')
ax.fill_between(S_range, call_payoff, alpha=0.1, color=COLORS['quantum'])
ax.set_xlabel('Stock price at expiry $S_T$')
ax.set_ylabel('Payoff ($)')
ax.set_title('Payoff at Expiry')
ax.legend(fontsize=9)

# ── Panel 2: BS price as function of spot ──────────────────────────────────
ax = axes[1]
call_prices = [black_scholes(s, K, T, r, sigma, 'call') for s in S_range]
put_prices  = [black_scholes(s, K, T, r, sigma, 'put')  for s in S_range]
intrinsic_c = np.maximum(S_range - K * np.exp(-r * T), 0)

ax.plot(S_range, call_prices, color=COLORS['quantum'],   lw=2, label='Call price (BS)')
ax.plot(S_range, put_prices,  color=COLORS['classical'], lw=2, label='Put price (BS)')
ax.plot(S_range, intrinsic_c, color='gray', ls='--', lw=1, label='Intrinsic value')
ax.axvline(S0, color='black', ls=':', alpha=0.5)
ax.scatter([S0], [bs_call], color=COLORS['quantum'], s=80, zorder=5)
ax.set_xlabel('Current Stock Price $S$')
ax.set_ylabel('Option Price ($)')
ax.set_title('Black-Scholes Price vs Spot')
ax.legend(fontsize=9)

# ── Panel 3: Vol smile / term structure ────────────────────────────────────
ax = axes[2]
sigma_range = np.linspace(0.05, 0.60, 100)
T_values = [0.25, 0.5, 1.0, 2.0]
for T_val in T_values:
    prices = [black_scholes(S0, K, T_val, r, s, 'call') for s in sigma_range]
    ax.plot(sigma_range * 100, prices, lw=2, label=f'T={T_val}y')
ax.axvline(sigma * 100, color='gray', ls='--', alpha=0.5, label=f'σ={sigma*100:.0f}%')
ax.set_xlabel('Implied Volatility (%)')
ax.set_ylabel('Call Price ($)')
ax.set_title('Price vs Volatility (Term Structure)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Black-Scholes Option Pricing — Baseline Reference',
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig('bs_option_pricing.png', bbox_inches='tight', dpi=120)
plt.show()
print(f"BS Call price at (S={S0}, K={K}, σ={sigma}): ${bs_call:.4f}")

## 2. Classical Monte Carlo — The Baseline

Classical Monte Carlo simulates many stock price paths and averages the
discounted payoff. The Central Limit Theorem gives:

$$\text{Error} \approx \frac{\sigma_{\text{payoff}}}{\sqrt{N}}$$

This **O(1/√N)** convergence is the rate that quantum algorithms improve upon.

In [ ]:
def mc_option_price(S, K, T, r, sigma, option='call', n_samples=100_000, seed=42):
    """
    Monte Carlo option pricing via GBM simulation.

    Under risk-neutral measure:
      S_T = S * exp((r - 0.5*σ²)*T + σ*√T*Z),  Z ~ N(0,1)

    Reference: Glasserman (2003) Monte Carlo Methods in Financial Engineering
    """
    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(n_samples)
    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    if option == 'call':
        payoffs = np.maximum(S_T - K, 0)
    else:
        payoffs = np.maximum(K - S_T, 0)

    price     = np.exp(-r * T) * np.mean(payoffs)
    std_error = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_samples)
    return price, std_error, payoffs


def mc_cvar(mu_annual, sigma_annual, S_0, alpha=0.05, T=1/252,
            n_samples=100_000, seed=42):
    """
    Classical Monte Carlo CVaR (Conditional Value at Risk).

    CVaR_α = E[Loss | Loss > VaR_α]

    This is the 'quantum risk analysis' target of Woerner & Egger (2019).
    Standard MC error: σ_loss / sqrt(N)
    """
    rng = np.random.default_rng(seed)
    daily_mu    = mu_annual / 252
    daily_sigma = sigma_annual / np.sqrt(252)

    Z = rng.standard_normal(n_samples)
    S_T = S_0 * np.exp((daily_mu - 0.5 * daily_sigma**2) * T +
                        daily_sigma * np.sqrt(T) * Z)
    losses = S_0 - S_T  # positive = loss

    var_alpha = np.percentile(losses, (1 - alpha) * 100)
    cvar_tail = losses[losses >= var_alpha]
    cvar      = np.mean(cvar_tail)
    cvar_std  = np.std(cvar_tail) / np.sqrt(len(cvar_tail))

    return {
        'var':      var_alpha,
        'cvar':     cvar,
        'cvar_std': cvar_std,
        'n_tail':   len(cvar_tail),
        'losses':   losses,
    }


# ── Run MC option pricing ──────────────────────────────────────────────────
mc_price, mc_se, payoffs = mc_option_price(S0, K, T, r, sigma, n_samples=100_000)
print(f"Monte Carlo Call:   ${mc_price:.4f}  ±{mc_se:.4f}  (1σ)")
print(f"Black-Scholes Call: ${bs_call:.4f}")
print(f"Error vs BS:        {abs(mc_price - bs_call):.4f}  ({abs(mc_price-bs_call)/bs_call*100:.2f}%)")

# ── Run MC CVaR ────────────────────────────────────────────────────────────
mu_stock = 0.08   # 8% annual drift
cvar_result = mc_cvar(mu_stock, sigma, S0, alpha=0.05, n_samples=200_000)
print(f"\n5% CVaR (1-day):    ${cvar_result['cvar']:.4f}  ±{cvar_result['cvar_std']:.4f}")
print(f"5% VaR  (1-day):    ${cvar_result['var']:.4f}")
print(f"Tail observations:  {cvar_result['n_tail']}")

## 3. Monte Carlo Convergence Rate

Here we empirically confirm the O(1/√N) convergence of classical MC.
This is the rate that Quantum Amplitude Estimation replaces with O(1/N).

In [ ]:
# ── Empirical MC convergence study ────────────────────────────────────────
sample_sizes = np.logspace(2, 6, 40).astype(int)
mc_errors    = []
mc_stderrs   = []

rng = np.random.default_rng(0)
for N in sample_sizes:
    Z   = rng.standard_normal(N)
    S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    p   = np.exp(-r * T) * np.maximum(S_T - K, 0)
    mc_errors.append(abs(np.mean(p) - bs_call))
    mc_stderrs.append(np.std(p) / np.sqrt(N))

mc_errors   = np.array(mc_errors)
mc_stderrs  = np.array(mc_stderrs)

# Fit power law: error ~ C / N^alpha
log_N = np.log(sample_sizes)
log_e = np.log(mc_stderrs + 1e-12)
slope, intercept = np.polyfit(log_N, log_e, 1)
fitted = np.exp(intercept) * sample_sizes**slope

print(f"Fitted convergence rate: O(1/N^{abs(slope):.3f})  [theory: O(1/N^0.5)]")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.loglog(sample_sizes, mc_stderrs, 'o-', color=COLORS['classical'],
          ms=4, lw=1.5, label='MC std error (empirical)')
ax.loglog(sample_sizes, fitted, '--', color='gray', lw=2,
          label=f'Fit: O(1/N^{abs(slope):.2f})')
ax.loglog(sample_sizes, mc_stderrs[0]*np.sqrt(sample_sizes[0]/sample_sizes),
          ':', color='red', lw=1.5, label='Theory: O(1/√N)')
ax.set_xlabel('Number of MC samples N')
ax.set_ylabel('Standard Error ($)')
ax.set_title('Classical MC Convergence')
ax.legend()

ax = axes[1]
ax.loglog(sample_sizes, mc_errors, 's-', color=COLORS['classical'],
          ms=4, lw=1.5, label='|MC price − BS price|')
ax.loglog(sample_sizes, 3*mc_stderrs, '--', color='orange', lw=1.5, label='3σ bound')
ax.set_xlabel('Number of MC samples N')
ax.set_ylabel('Absolute Error ($)')
ax.set_title('MC Price Error vs Sample Size')
ax.legend()

plt.tight_layout()
plt.savefig('mc_convergence.png', bbox_inches='tight', dpi=120)
plt.show()

## 4. Quantum Amplitude Estimation (QAE) — Theory

**Woerner & Egger (2019)** and **Rebentrost et al.** show that QAE achieves
**O(1/M)** precision using M quantum circuit evaluations, versus O(1/√N) for MC.

### The QAE Setup

Define a quantum operator $\mathcal{A}$ that encodes the option payoff:

$$\mathcal{A}|0\rangle = \sqrt{1-a}\,|\Psi_0\rangle|0\rangle + \sqrt{a}\,|\Psi_1\rangle|1\rangle$$

where $a$ is the **amplitude** encoding the expected normalized payoff.

### Grover-like Amplification

The Quantum Amplitude Estimation algorithm applies the operator
$Q = \mathcal{A}S_0\mathcal{A}^\dagger S_\chi$ repeatedly (M times),
then uses **Quantum Phase Estimation (QPE)** to extract $\theta$ where:

$$a = \sin^2(\theta)$$

### Precision Comparison

| Method | Evaluations | Precision |
|--------|------------|-----------|
| Classical MC | N | σ/√N |
| QAE (canonical) | M | π/(2M) |
| QAE speedup | — | **Quadratic** |

The crossover happens at $N^* = \left(\frac{\pi}{2\sigma}\right)^2$

In [ ]:
def qae_precision(M):
    """
    Canonical QAE precision bound.

    After M oracle calls, QAE achieves:
      |a_est - a_true| <= pi / (2*M)

    Reference: Brassard et al. (2002), Woerner & Egger (2019) Eq. 6
    """
    return np.pi / (2 * M)


def mc_precision(sigma_payoff, N):
    """
    Classical MC precision (1-sigma standard error).
    sigma_payoff: std of discounted payoff
    """
    return sigma_payoff / np.sqrt(N)


def qae_price_error(M, payoff_max):
    """
    Convert QAE amplitude precision to price precision.
    Payoffs are normalized to [0, payoff_max], so:
      price_error = payoff_max * qae_precision(M) * exp(-rT)
    """
    return payoff_max * qae_precision(M) * np.exp(-r * T)


# ── Payoff normalization ───────────────────────────────────────────────────
# For a call: max payoff ≈ S_max - K where S_max is upper truncation
S_max       = 200.0
payoff_max  = S_max - K   # ~95 for our parameters
sigma_payoff = np.std(payoffs)

# ── Precision vs evaluations ───────────────────────────────────────────────
evals = np.logspace(1, 5, 200).astype(int)

qae_price_errors = qae_price_error(evals, payoff_max)
mc_price_errors  = mc_precision(sigma_payoff, evals)

crossover_N = (np.pi / (2 * sigma_payoff))**2
crossover_price_err = mc_precision(sigma_payoff, crossover_N)

print(f"Payoff std deviation (σ_payoff):  ${sigma_payoff:.4f}")
print(f"Crossover point N*:               {crossover_N:.0f} evaluations")
print(f"Error at crossover:               ${crossover_price_err:.4f}")
print()
print(f"At N=1000 evaluations:")
print(f"  Classical MC error: ${mc_precision(sigma_payoff, 1000):.4f}")
print(f"  QAE error:          ${qae_price_error(1000, payoff_max):.4f}")
print(f"  QAE speedup:        {mc_precision(sigma_payoff,1000)/qae_price_error(1000,payoff_max):.1f}x")

## 5. Quadratic Speedup — Replicating Woerner & Egger (2019) Fig. 2

The central result: QAE error falls as **O(1/M)** vs classical MC at **O(1/√N)**.
This is the quadratic speedup that makes quantum computing attractive for finance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Panel 1: Main convergence comparison (replicates W&E Fig. 2) ──────────
ax = axes[0]
ax.loglog(evals, mc_price_errors,  '-',  color=COLORS['classical'], lw=2.5,
          label=r'Classical MC: $\mathcal{O}(1/\sqrt{N})$')
ax.loglog(evals, qae_price_errors, '-',  color=COLORS['quantum'],   lw=2.5,
          label=r'QAE: $\mathcal{O}(1/M)$')

# Mark crossover
ax.axvline(crossover_N, color='gray', ls='--', alpha=0.7)
ax.scatter([crossover_N], [crossover_price_err], s=120, zorder=5,
           color='gold', edgecolors='black', lw=1.5,
           label=f'Crossover N*≈{crossover_N:.0f}')

# Reference lines
ax.loglog(evals, 10*evals**(-0.5), ':', color=COLORS['classical'], lw=1, alpha=0.4)
ax.loglog(evals, 200*evals**(-1.0), ':', color=COLORS['quantum'],   lw=1, alpha=0.4)

ax.set_xlabel('Number of circuit evaluations / samples', fontsize=12)
ax.set_ylabel('Expected price error ($)', fontsize=12)
ax.set_title('QAE vs Classical MC Convergence\n(Woerner & Egger 2019, Fig. 2)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([10, 1e5])

# Annotate speedup region
ax.annotate('QAE advantage region', xy=(5000, 0.005), fontsize=9,
            color=COLORS['quantum'], style='italic')

# ── Panel 2: Speedup ratio ─────────────────────────────────────────────────
ax = axes[1]
speedup = mc_price_errors / qae_price_errors
ax.semilogx(evals, speedup, '-', color='purple', lw=2.5)
ax.axvline(crossover_N, color='gray', ls='--', alpha=0.7, label=f'Crossover ≈{crossover_N:.0f}')
ax.axhline(1, color='gray', ls='-', alpha=0.3)

# Fill above crossover
mask = evals > crossover_N
ax.fill_between(evals[mask], speedup[mask], 1, alpha=0.15, color='purple',
                label='QAE advantage')
ax.set_xlabel('Number of evaluations', fontsize=12)
ax.set_ylabel('Speedup factor (MC error / QAE error)', fontsize=12)
ax.set_title('QAE Speedup Factor vs Classical MC', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Annotate key points
for N_val in [1000, 10000, 100000]:
    s = mc_precision(sigma_payoff, N_val) / qae_price_error(N_val, payoff_max)
    ax.annotate(f'{s:.0f}x', xy=(N_val, s), xytext=(N_val*0.5, s+2),
                fontsize=8, color='purple')

plt.tight_layout()
plt.suptitle('Quantum Amplitude Estimation Advantage\n(Replicating Woerner & Egger 2019)',
             fontsize=13, fontweight='bold', y=1.02)
plt.savefig('qae_vs_mc_speedup.png', bbox_inches='tight', dpi=120)
plt.show()

print("\nKey result: At N=10,000 evaluations:")
print(f"  MC error:  ${mc_precision(sigma_payoff, 10000):.4f}")
print(f"  QAE error: ${qae_price_error(10000, payoff_max):.4f}")
print(f"  Speedup:   {mc_precision(sigma_payoff,10000)/qae_price_error(10000,payoff_max):.1f}x")

## 6. Quantum Circuit for Option Pricing (Stamatopoulos et al. 2020)

Stamatopoulos et al. (2020) show how to encode log-normal stock price
distributions and European option payoffs into quantum circuits.

### Encoding Strategy

**Step 1**: Discretize log-normal distribution into n qubits → 2ⁿ price levels
**Step 2**: Load probabilities via amplitude encoding
**Step 3**: Compute payoff via comparator + linear function circuits
**Step 4**: Apply QAE to extract expected payoff

We simulate this classically to validate the encoding.

In [ ]:
def encode_lognormal(S, r, sigma, T, n_qubits=5):
    """
    Discretize log-normal distribution into 2^n_qubits levels.

    Stamatopoulos et al. (2020) §2: Probability loading circuit
    Maps S_T distribution onto quantum state amplitudes.

    Returns: (price_levels, probabilities, amplitudes)
    """
    n_states  = 2**n_qubits
    S_min_log = np.log(S) + (r - 0.5*sigma**2)*T - 4*sigma*np.sqrt(T)
    S_max_log = np.log(S) + (r - 0.5*sigma**2)*T + 4*sigma*np.sqrt(T)

    log_prices   = np.linspace(S_min_log, S_max_log, n_states)
    price_levels = np.exp(log_prices)

    # Log-normal PDF in discretized form
    mu_log  = np.log(S) + (r - 0.5*sigma**2)*T
    sig_log = sigma * np.sqrt(T)

    log_probs = norm.pdf(log_prices, mu_log, sig_log)
    log_probs /= log_probs.sum()   # normalize

    # Quantum amplitudes = sqrt(probabilities)
    amplitudes = np.sqrt(log_probs)

    return price_levels, log_probs, amplitudes


def quantum_payoff_encoding(price_levels, K, option='call'):
    """
    Encode option payoff into quantum register.

    Stamatopoulos et al. (2020) §3: Payoff function circuit
    Uses comparator + linear amplitude function.

    Returns normalized payoffs in [0, 1]
    """
    if option == 'call':
        payoffs = np.maximum(price_levels - K, 0)
    else:
        payoffs = np.maximum(K - price_levels, 0)

    payoff_max = payoffs.max()
    if payoff_max > 0:
        normalized = payoffs / payoff_max
    else:
        normalized = payoffs

    return payoffs, normalized, payoff_max


def simulate_qae_price(price_levels, probabilities, payoffs, r, T, n_shots=1000):
    """
    Simulate QAE price estimation.

    In real hardware: QAE extracts amplitude a = sum_i p_i * f(S_i)
    We simulate by adding QAE-like shot noise.

    QAE noise model: epsilon ~ N(0, pi/(2*M)) in amplitude space
    """
    # True amplitude (what ideal QAE reads)
    true_amplitude = np.sum(probabilities * payoffs) / payoffs.max() if payoffs.max() > 0 else 0

    # QAE precision: pi/(2*n_shots) in amplitude
    qae_noise_std = np.pi / (2 * n_shots)
    noisy_amplitude = true_amplitude + np.random.normal(0, qae_noise_std)
    noisy_amplitude = np.clip(noisy_amplitude, 0, 1)

    # Convert back to price
    estimated_price = np.exp(-r * T) * noisy_amplitude * payoffs.max()
    true_price      = np.exp(-r * T) * np.sum(probabilities * payoffs)

    return estimated_price, true_price, true_amplitude


# ── Run encoding ────────────────────────────────────────────────────────────
n_qubits = 5
price_levels, probs, amplitudes = encode_lognormal(S0, r, sigma, T, n_qubits)
payoffs_disc, payoffs_norm, pmax = quantum_payoff_encoding(price_levels, K)

# QAE price estimate
qae_est, qae_true, amp_true = simulate_qae_price(
    price_levels, probs, payoffs_disc, r, T, n_shots=1000)

print(f"Encoding: {2**n_qubits} price levels from ${price_levels[0]:.1f} to ${price_levels[-1]:.1f}")
print(f"\nPricing results:")
print(f"  Black-Scholes (closed form):  ${bs_call:.4f}")
print(f"  Classical MC (100k samples):  ${mc_price:.4f}")
print(f"  Discretized QAE estimate:     ${qae_true:.4f}  (noiseless)")
print(f"  Simulated QAE (1000 shots):   ${qae_est:.4f}")
print(f"\nDiscretization error (n={n_qubits} qubits): {abs(qae_true - bs_call)/bs_call*100:.3f}%")
print(f"  → {2**n_qubits} price levels; use n≥8 qubits for <0.1% error")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Panel 1: Log-normal amplitude encoding ────────────────────────────────
ax = axes[0, 0]
ax.bar(range(len(probs)), amplitudes, color=COLORS['classical'], alpha=0.7,
       label='Quantum amplitudes √p_i')
ax.plot(range(len(probs)), probs / probs.max() * amplitudes.max(),
        color=COLORS['quantum'], lw=2, label='Normalized prob density')
ax.set_xlabel('Quantum basis state |i⟩')
ax.set_ylabel('Amplitude |α_i|')
ax.set_title(f'Amplitude Encoding of Log-Normal\n({2**n_qubits} states, {n_qubits} qubits)')
ax.legend(fontsize=9)

# ── Panel 2: Payoff encoding ───────────────────────────────────────────────
ax = axes[0, 1]
ax_twin = ax.twinx()
ax.bar(range(len(price_levels)), payoffs_disc, color=COLORS['cvar'], alpha=0.6, label='Call payoff $')
ax_twin.plot(range(len(price_levels)), payoffs_norm, color=COLORS['quantum'], lw=2,
             label='Normalized payoff f(S)')
ax.axvline(np.searchsorted(price_levels, K), color='gray', ls='--', alpha=0.7, label=f'Strike K={K}')
ax.set_xlabel('Quantum basis state |i⟩')
ax.set_ylabel('Payoff ($)', color=COLORS['cvar'])
ax_twin.set_ylabel('Normalized f(S) ∈ [0,1]', color=COLORS['quantum'])
ax.set_title('Payoff Function Encoding\n(Stamatopoulos et al. 2020)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

# ── Panel 3: QAE convergence for THIS option ──────────────────────────────
ax = axes[1, 0]
M_vals = np.logspace(1, 4, 50).astype(int)
qae_errs_here = [qae_price_error(m, pmax) for m in M_vals]
mc_errs_here  = [mc_precision(sigma_payoff, m) for m in M_vals]

ax.loglog(M_vals, mc_errs_here,  '-', color=COLORS['classical'], lw=2,
          label='Classical MC')
ax.loglog(M_vals, qae_errs_here, '-', color=COLORS['quantum'],   lw=2,
          label='QAE (Stamatopoulos)')
ax.axhline(0.01, color='gray', ls=':', label='$0.01 target')
ax.set_xlabel('Evaluations M (or samples N)')
ax.set_ylabel('Price error ($)')
ax.set_title(f'Convergence for Call(S={S0},K={K},T={T},σ={sigma})')
ax.legend(fontsize=9)

# ── Panel 4: Multi-qubit precision trade-off ──────────────────────────────
ax = axes[1, 1]
qubit_counts = range(3, 12)
disc_errors  = []
for nq in qubit_counts:
    pl, pb, _ = encode_lognormal(S0, r, sigma, T, nq)
    po, _, _  = quantum_payoff_encoding(pl, K)
    disc_price = np.exp(-r * T) * np.sum(pb * po)
    disc_errors.append(abs(disc_price - bs_call))

ax.semilogy(list(qubit_counts), disc_errors, 'o-', color=COLORS['quantum'], lw=2, ms=8)
ax.axhline(0.001, color='gray', ls=':', label='$0.001 (0.01%) target')
ax.axhline(0.01,  color='gray', ls='--', label='$0.01 (0.1%) target')
ax.set_xlabel('Number of qubits n')
ax.set_ylabel('Discretization error ($)')
ax.set_title('Discretization Error vs Qubit Count')
ax.legend(fontsize=9)
ax.set_xticks(list(qubit_counts))

plt.tight_layout()
plt.suptitle('Quantum Circuit Option Pricing (Stamatopoulos et al. 2020)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('quantum_circuit_pricing.png', bbox_inches='tight', dpi=120)
plt.show()

## 7. Multi-Asset CVaR — Woerner & Egger (2019) Core Application

The primary application in Woerner & Egger (2019) is CVaR estimation for
a **portfolio** of assets. QAE computes CVaR with the same O(1/M) speedup.

In [ ]:
# ── Portfolio definition ──────────────────────────────────────────────────
ASSETS = {
    'AAPL': {'mu': 0.18, 'sigma': 0.28, 'weight': 0.25},
    'MSFT': {'mu': 0.20, 'sigma': 0.25, 'weight': 0.25},
    'GOOGL': {'mu': 0.15, 'sigma': 0.24, 'weight': 0.20},
    'AMZN': {'mu': 0.22, 'sigma': 0.30, 'weight': 0.15},
    'META': {'mu': 0.17, 'sigma': 0.35, 'weight': 0.15},
}

# Correlation matrix (approximate market structure)
corr_matrix = np.array([
    [1.00, 0.72, 0.65, 0.58, 0.55],
    [0.72, 1.00, 0.68, 0.62, 0.60],
    [0.65, 0.68, 1.00, 0.63, 0.58],
    [0.58, 0.62, 0.63, 1.00, 0.52],
    [0.55, 0.60, 0.58, 0.52, 1.00],
])

asset_names  = list(ASSETS.keys())
weights      = np.array([ASSETS[a]['weight'] for a in asset_names])
mus          = np.array([ASSETS[a]['mu']     for a in asset_names])
sigmas       = np.array([ASSETS[a]['sigma']  for a in asset_names])
cov_matrix   = np.outer(sigmas, sigmas) * corr_matrix

portfolio_mu    = weights @ mus
portfolio_sigma = np.sqrt(weights @ cov_matrix @ weights)

print("Multi-Asset Portfolio:")
print(f"  Assets: {', '.join(asset_names)}")
print(f"  Portfolio annual μ:  {portfolio_mu:.3f} ({portfolio_mu*100:.1f}%)")
print(f"  Portfolio annual σ:  {portfolio_sigma:.3f} ({portfolio_sigma*100:.1f}%)")
print()


def multi_asset_cvar(weights, mus, cov_matrix, S0=100, alpha=0.05,
                     horizon_days=1, n_samples=500_000, seed=0):
    """
    Multi-asset CVaR via correlated MC simulation.
    Uses Cholesky decomposition for correlation structure.
    """
    rng = np.random.default_rng(seed)
    T_h = horizon_days / 252

    # Daily parameters
    mu_d  = mus / 252
    cov_d = cov_matrix / 252

    # Cholesky for correlated sampling
    L = np.linalg.cholesky(cov_d)
    Z = rng.standard_normal((n_samples, len(weights)))

    # Correlated returns
    returns_sim = mu_d * T_h + (L @ Z.T).T * np.sqrt(T_h)

    # Portfolio P&L
    portfolio_returns = returns_sim @ weights
    portfolio_pnl     = S0 * portfolio_returns  # dollar P&L
    losses            = -portfolio_pnl          # positive = loss

    var_alpha  = np.percentile(losses, (1-alpha)*100)
    cvar_alpha = losses[losses >= var_alpha].mean()

    return {
        'var_1d_5pct':  var_alpha,
        'cvar_1d_5pct': cvar_alpha,
        'annual_sharpe': (portfolio_returns.mean() * 252) / (portfolio_returns.std() * np.sqrt(252)),
    }


# ── Run for multiple alpha levels ─────────────────────────────────────────
alpha_levels = [0.01, 0.025, 0.05, 0.10]
cvar_results = {}
for alpha in alpha_levels:
    cvar_results[alpha] = multi_asset_cvar(weights, mus, cov_matrix, alpha=alpha)

# ── Summary table ─────────────────────────────────────────────────────────
print("\nMulti-Asset CVaR Table (1-day horizon, $100 portfolio):")
print(f"{'Alpha':<10} {'VaR ($)':<12} {'CVaR ($)':<12} {'CVaR/VaR':<10}")
print("-" * 45)
for alpha in alpha_levels:
    v = cvar_results[alpha]['var_1d_5pct']
    c = cvar_results[alpha]['cvar_1d_5pct']
    print(f"{alpha*100:>6.1f}%    ${v:>8.3f}    ${c:>8.3f}    {c/v:>6.2f}x")

print(f"\nPortfolio Sharpe (simulated): {cvar_results[0.05]['annual_sharpe']:.3f}")

In [ ]:
# ── Full CVaR visualization ────────────────────────────────────────────────
result_5pct = multi_asset_cvar(weights, mus, cov_matrix, alpha=0.05, n_samples=200_000)

# Reconstruct losses for plotting
rng = np.random.default_rng(0)
T_h = 1/252
mu_d = mus / 252; cov_d = cov_matrix / 252
L = np.linalg.cholesky(cov_d)
Z = rng.standard_normal((200_000, len(weights)))
returns_sim = mu_d * T_h + (L @ Z.T).T * np.sqrt(T_h)
portfolio_returns = returns_sim @ weights
losses = -100 * portfolio_returns

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Panel 1: Loss distribution ─────────────────────────────────────────────
ax = axes[0]
var_val  = np.percentile(losses, 95)
cvar_val = losses[losses >= var_val].mean()

counts, bin_edges = np.histogram(losses, bins=150, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
ax.fill_between(bin_centers, 0, counts,
                where=(bin_centers >= var_val),
                color=COLORS['cvar'], alpha=0.6, label=f'CVaR tail (α=5%)')
ax.fill_between(bin_centers, 0, counts,
                where=(bin_centers < var_val),
                color=COLORS['classical'], alpha=0.3, label='Non-tail losses')
ax.axvline(var_val,  color=COLORS['cvar'],     lw=2, ls='--', label=f'VaR  = ${var_val:.2f}')
ax.axvline(cvar_val, color=COLORS['classical'], lw=2, ls='-',  label=f'CVaR = ${cvar_val:.2f}')
ax.set_xlabel('1-Day Loss ($)')
ax.set_ylabel('Probability Density')
ax.set_title('Portfolio Loss Distribution')
ax.legend(fontsize=8)

# ── Panel 2: CVaR vs alpha ─────────────────────────────────────────────────
ax = axes[1]
alphas  = np.linspace(0.01, 0.20, 40)
vars_v  = [np.percentile(losses, (1-a)*100) for a in alphas]
cvars_v = [losses[losses >= np.percentile(losses, (1-a)*100)].mean() for a in alphas]

ax.plot(alphas*100, vars_v,  '-', color=COLORS['classical'], lw=2, label='VaR')
ax.plot(alphas*100, cvars_v, '-', color=COLORS['cvar'],      lw=2, label='CVaR')
ax.fill_between(alphas*100, vars_v, cvars_v, alpha=0.15, color=COLORS['cvar'],
                label='CVaR excess')
ax.set_xlabel('Confidence Level α (%)')
ax.set_ylabel('Risk Measure ($)')
ax.set_title('CVaR vs VaR by Confidence Level')
ax.legend()

# ── Panel 3: QAE vs MC precision for CVaR ─────────────────────────────────
ax = axes[2]
sigma_loss = np.std(losses[losses >= var_val])
M_range    = np.logspace(2, 5, 100).astype(int)
mc_cvar_err  = sigma_loss / np.sqrt(M_range * 0.05)  # only 5% in tail
qae_cvar_err = np.pi / (2 * M_range) * 100           # amplitude → dollar

ax.loglog(M_range, mc_cvar_err,  '-', color=COLORS['classical'], lw=2,
          label='Classical MC CVaR error')
ax.loglog(M_range, qae_cvar_err, '-', color=COLORS['quantum'],   lw=2,
          label='QAE CVaR error')
ax.set_xlabel('Circuit evaluations / samples')
ax.set_ylabel('CVaR estimation error ($)')
ax.set_title('QAE vs MC for CVaR\n(Woerner & Egger 2019 application)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.suptitle(f'Multi-Asset CVaR: {", ".join(asset_names)}',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('multi_asset_cvar.png', bbox_inches='tight', dpi=120)
plt.show()

## 8. Summary — Quantum Risk Analysis

| Concept | Classical | Quantum (QAE) | Speedup |
|---------|-----------|---------------|---------|
| MC option pricing | O(1/√N) | O(1/M) | **Quadratic** |
| CVaR estimation | O(1/√N) | O(1/M) | **Quadratic** |
| Circuit depth | — | O(n) qubits | — |
| Discretization | N/A | 2^n levels | n qubits needed |

### Key References
1. **Woerner & Egger (2019)** — First demonstration of QAE for financial risk
2. **Stamatopoulos et al. (2020)** — Full quantum circuit for European option pricing
3. **Rebentrost et al.** — Monte Carlo pricing via quantum speedup

### Next: NB04 — QUBO + VQE Portfolio Optimization
Building on these speedup concepts, NB04 applies quantum/variational methods
directly to portfolio weight selection via combinatorial optimization.

In [ ]:
# ── Final summary printout ─────────────────────────────────────────────────
print("=" * 60)
print("NB03 SUMMARY: Quantum Risk Analysis & Option Pricing")
print("=" * 60)
print()
print("1. Black-Scholes Reference:")
print(f"   Call(S={S0},K={K},T={T},σ={sigma}): ${bs_call:.4f}")
print()
print("2. Classical MC convergence:")
print(f"   Error rate: O(1/√N) ≈ σ/√N")
print(f"   σ_payoff = ${sigma_payoff:.4f}")
print()
print("3. QAE convergence:")
print(f"   Error rate: O(1/M) = π/(2M)")
print(f"   Crossover N* ≈ {crossover_N:.0f}")
print()
print("4. Multi-asset portfolio:")
print(f"   Assets: {', '.join(asset_names)}")
print(f"   1-day 5% VaR:  ${var_val:.3f}")
print(f"   1-day 5% CVaR: ${cvar_val:.3f}")
print()
print("Key insight: QAE provides quadratic speedup over Monte Carlo.")
print("For N=1M samples, QAE with M=1000 evaluations achieves same precision.")
print()
print("Papers implemented:")
print("  ✓ Woerner & Egger (2019) — QAE for CVaR")
print("  ✓ Stamatopoulos et al. (2020) — Quantum circuit option pricing")
print("  ✓ Rebentrost et al. — Monte Carlo quantum speedup")